In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [2]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [23]:

esquemaSales = "Sales"
tablaSalesTerritory = pd.read_sql_table("SalesTerritory", motorBaseDatos, esquemaSales)
tablaSalesTerritory


c:\Users\dange.DANGERPC\OneDrive\Escritorio\etl-aventure-works\my_env\Lib\site-packages\pandas\io\sql.py:1737: SAWarning: Did not recognize type 'Name' of column 'Name'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


,TerritoryID,Name,CountryRegionCode,Group,SalesYTD,SalesLastYear,CostYTD,CostLastYear,rowguid,ModifiedDate
0,1,Northwest,US,North America,7.887187e+06,3.298694e+06,0.0,0.0,43689a10-e30b-497f-b0de-11de20267ff7,2008-04-30
1,2,Northeast,US,North America,2.402177e+06,3.607149e+06,0.0,0.0,00fb7309-96cc-49e2-8363-0a1ba72486f2,2008-04-30
2,3,Central,US,North America,3.072175e+06,3.205014e+06,0.0,0.0,df6e7fd8-1a8d-468c-b103-ed8addb452c1,2008-04-30
3,4,Southwest,US,North America,1.051085e+07,5.366576e+06,0.0,0.0,dc3e9ea0-7950-4431-9428-99dbcbc33865,2008-04-30
4,5,Southeast,US,North America,2.538667e+06,3.925071e+06,0.0,0.0,6dc4165a-5e4c-42d2-809d-4344e0ac75e7,2008-04-30
5,6,Canada,CA,North America,6.771829e+06,5.693989e+06,0.0,0.0,06b4af8a-1639-476e-9266-110461d66b00,2008-04-30
6,7,France,FR,Europe,4.772398e+06,2.396540e+06,0.0,0.0,bf806804-9b4c-4b07-9d19-706f2e689552,2008-04-30
7,8,Germany,DE,Europe,3.805202e+06,1.307950e+06,0.0,0.0,6d2450db-8159-414f-a917-e73ee91c38a9,2008-04-30
8,9,Australia,AU,Pacific,5.977815e+06,2.278549e+06,0.0,0.0,602e612e-dfe9-41d9-b894-27e489747885,2008-04-30
9,10,United Kingdom,GB,Europe,5.012905e+06,1.635823e+06,0.0,0.0,05fc7e1f-2dea-414e-9ecd-09d150516fb5,2008-04-30


In [24]:
esquemaPerson = "Person"
tablaCountryRegion = pd.read_sql_table("CountryRegion", motorBaseDatos, esquemaPerson)
tablaCountryRegion

c:\Users\dange.DANGERPC\OneDrive\Escritorio\etl-aventure-works\my_env\Lib\site-packages\pandas\io\sql.py:1737: SAWarning: Did not recognize type 'Name' of column 'Name'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


,CountryRegionCode,Name,ModifiedDate
0,AD,Andorra,2008-04-30
1,AE,United Arab Emirates,2008-04-30
2,AF,Afghanistan,2008-04-30
3,AG,Antigua and Barbuda,2008-04-30
4,AI,Anguilla,2008-04-30
...,...,...,...
233,YE,Yemen,2008-04-30
234,YT,Mayotte,2008-04-30
235,ZA,South Africa,2008-04-30
236,ZM,Zambia,2008-04-30


TRANSFORMACION

In [25]:
dimensionSalesTerritory = tablaSalesTerritory



dimensionSalesTerritory.rename(columns={
    'TerritoryID'  : 'SalesTerritoryKey',
    'Name' : 'SalesTerritoryRegion',
    'Group' : 'SalesTerritoryGroup',
}, inplace=True)

dimensionSalesTerritory["SalesTerritoryAlternateKey"] = dimensionSalesTerritory["SalesTerritoryKey"]


dimensionSalesTerritory["SalesTerritoryImage"] = None


dimensionSalesTerritory.drop(columns=[
    'SalesYTD',
    'SalesLastYear',
    'CostYTD',
    'CostLastYear',
    'rowguid',
    'ModifiedDate'
], inplace=True)



dimensionSalesTerritory


,SalesTerritoryKey,SalesTerritoryRegion,CountryRegionCode,SalesTerritoryGroup,SalesTerritoryAlternateKey,SalesTerritoryImage
0,1,Northwest,US,North America,1,None
1,2,Northeast,US,North America,2,None
2,3,Central,US,North America,3,None
3,4,Southwest,US,North America,4,None
4,5,Southeast,US,North America,5,None
5,6,Canada,CA,North America,6,None
6,7,France,FR,Europe,7,None
7,8,Germany,DE,Europe,8,None
8,9,Australia,AU,Pacific,9,None
9,10,United Kingdom,GB,Europe,10,None


In [26]:
dimensionSalesTerritory = dimensionSalesTerritory.merge(tablaCountryRegion, on='CountryRegionCode')


dimensionSalesTerritory.rename(columns={
'Name' : 'SalesTerritoryCountry'

}, inplace=True)


dimensionSalesTerritory.drop(columns=[
    'CountryRegionCode',
    'ModifiedDate',
], inplace=True)

dimensionSalesTerritory

,SalesTerritoryKey,SalesTerritoryRegion,SalesTerritoryGroup,SalesTerritoryAlternateKey,SalesTerritoryImage,SalesTerritoryCountry
0,1,Northwest,North America,1,None,United States
1,2,Northeast,North America,2,None,United States
2,3,Central,North America,3,None,United States
3,4,Southwest,North America,4,None,United States
4,5,Southeast,North America,5,None,United States
5,6,Canada,North America,6,None,Canada
6,7,France,Europe,7,None,France
7,8,Germany,Europe,8,None,Germany
8,9,Australia,Pacific,9,None,Australia
9,10,United Kingdom,Europe,10,None,United Kingdom


CARGAR A LA BODEGA

In [27]:
dimensionSalesTerritory.to_sql('dimensionSalesTerritory',motorBodegaDatos, if_exists='replace',index=False)

10